In [0]:
# ── CÉLULA 1: configuração ───────────────────────────────────────────
BRONZE_PATH = "/Volumes/workspace/default/bronze/"
SILVER_PATH = "/Volumes/workspace/default/silver/"
GOLD_PATH   = "/Volumes/workspace/default/gold/"

print("Config OK")

In [0]:
# ── CÉLULA 2: leitura do Silver ──────────────────────────────────────
from pyspark.sql.functions import col, sum, count, round, avg, max

silver_clientes = spark.read.format("delta").load(f"{SILVER_PATH}clientes/")
silver_produtos  = spark.read.format("delta").load(f"{SILVER_PATH}produtos/")
silver_pedidos   = spark.read.format("delta").load(f"{SILVER_PATH}pedidos/")

print(f"Silver clientes: {silver_clientes.count()}")
print(f"Silver produtos: {silver_produtos.count()}")
print(f"Silver pedidos:  {silver_pedidos.count()}")

In [0]:
# ── CÉLULA 3: Gold — vendas por cliente ──────────────────────────────
gold_vendas_cliente = silver_pedidos \
    .filter(col("status") == "APROVADO") \
    .join(silver_clientes, "cliente_id") \
    .groupBy("cliente_id", "nome", "segmento", "cidade") \
    .agg(
        round(sum("valor_total"), 2).alias("total_vendas"),
        count("pedido_id").alias("qtd_pedidos"),
        round(avg("valor_total"), 2).alias("ticket_medio"),
        max("data_pedido").alias("ultimo_pedido")
    ) \
    .orderBy(col("total_vendas").desc())

gold_vendas_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}vendas_por_cliente/")

print(f"Gold vendas por cliente: {gold_vendas_cliente.count()} linhas")
display(gold_vendas_cliente)

In [0]:
# ── CÉLULA 4: Gold — vendas por categoria ────────────────────────────
gold_vendas_categoria = silver_pedidos \
    .filter(col("status") == "APROVADO") \
    .join(silver_produtos, "produto_id") \
    .groupBy("categoria") \
    .agg(
        round(sum("valor_total"), 2).alias("total_vendas"),
        count("pedido_id").alias("qtd_pedidos"),
        round(avg("valor_total"), 2).alias("ticket_medio")
    ) \
    .orderBy(col("total_vendas").desc())

gold_vendas_categoria.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}vendas_por_categoria/")

print(f"Gold vendas por categoria: {gold_vendas_categoria.count()} linhas")
display(gold_vendas_categoria)

In [0]:
# ── CÉLULA 5: Gold — ranking mensal de vendas ────────────────────────
from pyspark.sql.functions import date_format

gold_vendas_mensal = silver_pedidos \
    .filter(col("status") == "APROVADO") \
    .withColumn("mes", date_format(col("data_pedido"), "yyyy-MM")) \
    .groupBy("mes") \
    .agg(
        round(sum("valor_total"), 2).alias("total_vendas"),
        count("pedido_id").alias("qtd_pedidos")
    ) \
    .orderBy("mes")

gold_vendas_mensal.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_PATH}vendas_mensais/")

print(f"Gold vendas mensais: {gold_vendas_mensal.count()} linhas")
display(gold_vendas_mensal)

In [0]:
# ── CÉLULA 6: registrar no metastore como tabelas gerenciadas ────────
BRONZE_PATH = "/Volumes/workspace/default/bronze/"
SILVER_PATH = "/Volumes/workspace/default/silver/"
GOLD_PATH   = "/Volumes/workspace/default/gold/"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.lakehouse")

# lê os Delta e salva como tabelas gerenciadas no Unity Catalog
spark.read.format("delta").load(f"{BRONZE_PATH}clientes/") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.lakehouse.bronze_clientes")

spark.read.format("delta").load(f"{SILVER_PATH}pedidos/") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.lakehouse.silver_pedidos")

spark.read.format("delta").load(f"{GOLD_PATH}vendas_por_cliente/") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.lakehouse.gold_vendas_cliente")

spark.read.format("delta").load(f"{GOLD_PATH}vendas_por_categoria/") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.lakehouse.gold_vendas_categoria")

spark.read.format("delta").load(f"{GOLD_PATH}vendas_mensais/") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.lakehouse.gold_vendas_mensais")

print("Metastore atualizado!")

In [0]:
%sql
SHOW TABLES IN workspace.lakehouse;